In [1]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


In [2]:
import rag as rag

# Look at the first chunk we have in the vector index, take the patient_id attached to that chunk:
pid = rag.vector_documents[0]["patient_id"]

result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=pid,
    search_type="hybrid",
    num_results=5,
)

len(result["search_results"]), result["search_results"][:2]

(5,
 [{'id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'chunk_id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': 'eacfd84f2024e811caae390e056a4e52fefc5249',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Aurora248 Dooley940',
   'heading': 'Oncology Timeline: Aurora248 Dooley940',
   'chunk_text': '- Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': 'f50d5c213d71cb54157be1dbe78d05ff2cb23489',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': '304ddea3daed73b3c0c0f47535fef1906ea22987',
   'doc_type': 'oncology_timeline_events',
   'title': 'oncology_timeline_events.csv',
   'heading': 'oncology_timeline_events',
   'chunk_text': "event_type: Observation; date: 2015-10-01T05:29:59-04:00; label:

In [3]:
print("Answer cost (USD):", result["answer_total_cost_usd"])
print("Eval cost (USD):  ", result["eval_total_cost_usd"])
print("Overall cost (USD):", result["overall_total_cost_usd"])

print("Answer tokens (in/out/total):",
      result["prompt_tokens"],
      result["completion_tokens"],
      result["total_tokens"])

print("Eval tokens (in/out/total):",
      result["eval_prompt_tokens"],
      result["eval_completion_tokens"],
      result["eval_total_tokens"])

Answer cost (USD): 0.00315075
Eval cost (USD):   0.0022897499999999997
Overall cost (USD): 0.005440499999999999
Answer tokens (in/out/total): 1951 375 2326
Eval tokens (in/out/total): 2453 100 2553


In [4]:
for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc.get("chunk_id"))
    print("Patient ID:", doc.get("patient_id"))
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Text:", doc.get("chunk_text", "")[:500])

Rank: 1
Chunk ID: 6e3f2f8c188a50ec8d84a45b499a14a631fcc414
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Dooley940
Heading: Oncology Timeline: Aurora248 Dooley940
Text: - Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
- Oncology-related dated events: 52
Rank: 2
Chunk ID: f50d5c213d71cb54157be1dbe78d05ff2cb23489
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Text: event_type: Observation; date: 2015-10-01T05:29:59-04:00; label: Cancer Disease Progression; status: Patient's condition improved; resource_id: 3ee3c67c-a4a3-37e1-e49b-f20f32cc14db; source_file: data/prototype/sample50/Aurora248_Dooley940_03b93198-d95e-c385-c3a7-80470f411d18.json
Rank: 3
Chunk ID: 561c5809a7e9251705306dee52b08e9fd64721ce
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Doole

In [5]:
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
len(available_patient_ids), available_patient_ids[:10]

(50,
 ['03b93198-d95e-c385-c3a7-80470f411d18',
  '0c0f2095-e8ab-7ac4-6ef4-625748255480',
  '0f5704ee-b38b-5a68-449d-9c44806517d0',
  '188e1f01-15b7-d51b-c76d-bdd7772a10e9',
  '25197dc8-9425-1999-5914-f2171b0d4e32',
  '263375ec-5856-81b8-9e51-1cb8e8bcba30',
  '29f6beee-162f-0113-7884-72245814693f',
  '397b2de6-ccd8-858f-bf4a-b6fc379589bd',
  '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
  '3af995f1-02a5-07ee-5a7e-e2470a017f1e'])

# 9 patients for ground truth

We pick 9 patients with 3 from each complexity bucket (low, medium, high) for the ground-truth set, in order to cover simple, medium, and complex EHRs.

`n_resources` is the total number of FHIR resources in that patient’s bundle — i.e., how many individual clinical records (Patient, Encounter, Observation, Condition, Procedure, MedicationRequest, DiagnosticReport, etc.) are contained in the JSON file for that patient.

So we see that the higher `n_resources`, the higher `complexity_score`

Remember:
### Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



In [6]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [7]:
df

,filename,patient_id,patient_name,n_resources,n_encounters,n_observations,n_conditions,n_procedures,n_medication_requests,n_medication_administrations,n_diagnostic_reports,first_date,last_date,followup_days,complexity_score,complexity_bucket,sample_seed
0,data/raw/longitudinalMCODEBreast/Corrie32_Boyl...,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,230,27,80,2,27,20,0,28,2020-12-18T03:22:18-05:00,2022-05-20T18:36:55-04:00,518,317,low,42
1,data/raw/longitudinalMCODEBreast/Florine959_St...,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,261,25,119,5,23,11,0,26,2019-06-11T21:25:37-04:00,2022-06-21T05:03:45-04:00,1105,330,low,42
2,data/raw/longitudinalMCODEBreast/Deana43_Baumb...,3f130449-d7db-f118-5bd0-cce81084e911,Deana43 Baumbach677,431,42,206,10,23,18,0,44,2012-12-18T03:09:54-05:00,2022-04-12T04:24:54-04:00,3402,540,low,42
3,data/raw/longitudinalMCODEBreast/Joni720_Stied...,43c173b0-172c-f414-5c62-1bdf4bb33954,Joni720 Stiedemann542,459,47,218,17,29,5,0,51,2014-04-22T07:35:47-04:00,2022-05-17T21:08:31-04:00,2947,579,low,42
4,data/raw/longitudinalMCODEBreast/Santos184_Jas...,64ae3769-65e4-222e-6793-1a3bc14ec682,Santos184 Jaskolski867,468,45,241,4,31,11,0,48,2010-08-02T09:36:48-04:00,2022-05-17T22:36:39-04:00,4306,584,low,42
5,data/raw/longitudinalMCODEBreast/Mónica985_Se...,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,436,61,151,6,54,8,0,62,2018-01-07T15:16:05-05:00,2022-04-21T02:52:37-04:00,1564,602,low,42
6,data/raw/longitudinalMCODEBreast/Maryellen651_...,af3bd539-de27-28d9-9016-f1643d4615c0,Maryellen651 Zboncak558,514,54,238,11,35,18,0,56,2011-08-29T07:05:19-04:00,2022-04-27T18:02:46-04:00,3894,660,low,42
7,data/raw/longitudinalMCODEBreast/Darcie474_Fra...,568ec0af-94fa-521b-012e-88f61f78028f,Darcie474 Frami345,496,69,180,2,59,11,0,71,2015-10-27T05:01:54-04:00,2022-04-25T17:02:50-04:00,2372,686,low,42
8,data/raw/longitudinalMCODEBreast/Jani266_Thiel...,3e693a9a-de55-de2e-aab9-500036bcf04b,Jani266 Thiel172,629,76,258,8,67,14,0,80,2011-05-02T16:40:40-04:00,2022-06-03T06:55:38-04:00,4049,844,low,42
9,data/raw/longitudinalMCODEBreast/Dolores502_Ca...,6fb374e8-33aa-a5ea-f050-b61394dfcb99,Dolores502 Caldera106,872,101,305,13,165,18,1,111,2006-05-18T17:21:05-04:00,2022-06-30T17:36:05-04:00,5887,1244,low,42


In [8]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


In [9]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [ ]:
# looks at doc_type counts for selected patients, including oncology-flagged chunks

import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

db_path = Path("../data/retrieval/metadata.db")
conn = sqlite3.connect(db_path)

# Build a parameterized IN clause for SQLite
placeholders = ",".join(["?"] * len(selected_patient_ids))

doc_type_counts_selected = pd.read_sql_query(
    f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        COUNT(*) AS n_chunks,
        SUM(CASE WHEN chunks.is_oncology = 1 THEN 1 ELSE 0 END) AS n_oncology_chunks
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    GROUP BY chunks.patient_id, documents.doc_type
    ORDER BY chunks.patient_id, n_chunks DESC
    """,
    conn,
    params=selected_patient_ids,
)

conn.close()

print("Long format: one row per selected patient/doc_type")
display(doc_type_counts_selected)

# Wide pivot: rows = selected patients, columns = doc_types, values = total chunk counts
doc_type_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_chunks")
    .fillna(0)
    .astype(int)
)

print("\nChunk counts by doc_type for each selected patient")
display(doc_type_pivot_selected)

# Oncology-only pivot
oncology_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_oncology_chunks")
    .fillna(0)
    .astype(int)
)

print("\nOncology-flagged chunk counts by doc_type for each selected patient")
display(oncology_pivot_selected)

# List of doc_types present per selected patient
doc_types_per_selected_patient = (
    doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
    .groupby("patient_id")["doc_type"]
    .apply(list)
    .reset_index(name="doc_types")
)

print("\nDoc types present for each selected patient")
display(doc_types_per_selected_patient)

Long format: one row per selected patient/doc_type


,patient_id,doc_type,n_chunks,n_oncology_chunks
0,29f6beee-162f-0113-7884-72245814693f,observations,571,14
1,29f6beee-162f-0113-7884-72245814693f,procedures,408,37
2,29f6beee-162f-0113-7884-72245814693f,diagnostic_reports,284,0
3,29f6beee-162f-0113-7884-72245814693f,encounters,231,0
4,29f6beee-162f-0113-7884-72245814693f,oncology_timeline_events,53,53
...,...,...,...,...
76,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline_events,22,22
77,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,11,0
78,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,9,9
79,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,5,5



Chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,42,284,231,17,571,9,53,9,408
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,95,417,273,119,1133,6,26,9,651
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,56,250,197,41,565,9,49,9,465
4736727e-63f4-071a-1516-a49310f5a052,6,62,61,8,151,9,47,9,54
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,50,336,287,45,597,9,52,9,500
d65197b3-056a-2136-b584-77f43c29da3f,2,28,27,20,80,6,29,9,27
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,115,394,262,72,970,9,48,9,760
f203e11d-5573-1624-69b8-af8436987b3e,95,404,270,40,1322,9,52,9,791
f3739580-797d-ae04-eebf-aeddb2fc2f64,5,26,25,11,119,5,22,9,23



Oncology-flagged chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,2,0,0,0,14,9,53,9,37
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,2,0,0,0,14,6,26,0,10
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,2,0,0,0,11,9,49,9,36
4736727e-63f4-071a-1516-a49310f5a052,1,0,0,0,11,9,47,9,35
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,2,0,0,0,14,9,52,9,36
d65197b3-056a-2136-b584-77f43c29da3f,1,0,0,0,11,6,29,9,17
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,2,0,0,0,10,9,48,9,36
f203e11d-5573-1624-69b8-af8436987b3e,2,0,0,0,14,9,52,0,36
f3739580-797d-ae04-eebf-aeddb2fc2f64,2,0,0,0,10,5,22,9,10



Doc types present for each selected patient


,patient_id,doc_types
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,..."
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,..."
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,..."
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,..."


In [ ]:
# check that all selected patients have the same set of doc_types (ignoring order)

import pandas as pd
from IPython.display import display

# Example: doc_types_per_selected_patient already computed, like:
# doc_types_per_selected_patient = (
#     doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
#     .groupby("patient_id")["doc_type"]
#     .apply(list)
#     .reset_index(name="doc_types")
# )

# print("Doc types per selected patient:")
# display(doc_types_per_selected_patient)

# Convert each list of doc_types into a frozenset so order doesn't matter
doc_types_per_selected_patient["doc_types_set"] = (
    doc_types_per_selected_patient["doc_types"]
    .apply(lambda lst: frozenset(lst))
)

# Use the first patient's set as reference
reference_set = doc_types_per_selected_patient["doc_types_set"].iloc[0]
print("\nReference doc_types_set:", reference_set)

# Check equality against the reference
doc_types_per_selected_patient["matches_reference"] = (
    doc_types_per_selected_patient["doc_types_set"] == reference_set
)

print("\nEquality check vs reference:")
display(doc_types_per_selected_patient[["patient_id", "doc_types", "matches_reference"]])

# Summarize: how many distinct doc_types sets exist?
unique_sets = doc_types_per_selected_patient["doc_types_set"].unique()
print("\nNumber of distinct doc_types sets among selected patients:", len(unique_sets))

if len(unique_sets) > 1:
    print("Patients grouped by their doc_types_set:")
    # For readability, show each unique set and which patients have it
    for s in unique_sets:
        patients_with_set = doc_types_per_selected_patient[
            doc_types_per_selected_patient["doc_types_set"] == s
        ]["patient_id"].tolist()
        print(f"\nSet: {sorted(list(s))}")
        print("Patients:", patients_with_set)


Reference doc_types_set: frozenset({'oncology_timeline', 'procedures', 'observations', 'diagnostic_reports', 'encounters', 'oncology_timeline_events', 'medications', 'conditions', 'patient_overview'})

Equality check vs reference:


,patient_id,doc_types,matches_reference
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,...",True
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,...",True
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,...",True
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,...",True
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,...",True
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn...",True
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,...",True
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,...",True
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,...",True



Number of distinct doc_types sets among selected patients: 1


In [12]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of rows: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


In [13]:
onc_chunks["doc_type"].value_counts()

doc_type
oncology_timeline_events    378
procedures                  253
observations                109
oncology_timeline            71
patient_overview             63
conditions                   16
Name: count, dtype: int64

In [14]:
onc_chunks.groupby("patient_id")["doc_type"].count()

patient_id
29f6beee-162f-0113-7884-72245814693f    124
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     58
41681ed6-efc5-94c0-1bc0-f60b34dbd31b    116
4736727e-63f4-071a-1516-a49310f5a052    112
aee216e6-cbe8-eaf2-3241-4bd1e8a01494    122
d65197b3-056a-2136-b584-77f43c29da3f     73
ecc4a7d0-8838-36b4-44ba-676d5a1f7927    114
f203e11d-5573-1624-69b8-af8436987b3e    113
f3739580-797d-ae04-eebf-aeddb2fc2f64     58
Name: doc_type, dtype: int64

In [15]:
onc_chunks.groupby("doc_type")["patient_id"].value_counts()

doc_type                  patient_id                          
conditions                29f6beee-162f-0113-7884-72245814693f     2
                          ecc4a7d0-8838-36b4-44ba-676d5a1f7927     2
                          41681ed6-efc5-94c0-1bc0-f60b34dbd31b     2
                          f203e11d-5573-1624-69b8-af8436987b3e     2
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494     2
                          f3739580-797d-ae04-eebf-aeddb2fc2f64     2
                          3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     2
                          d65197b3-056a-2136-b584-77f43c29da3f     1
                          4736727e-63f4-071a-1516-a49310f5a052     1
observations              3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678    14
                          f203e11d-5573-1624-69b8-af8436987b3e    14
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494    14
                          29f6beee-162f-0113-7884-72245814693f    14
                          41681ed6-efc5-

In [16]:
chunks_df.columns

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

# Shared set of question

That’s a very good sign: it means all 9 selected patients have the **same set of document types** (even if the order in the lists differs).

Practically, this gives you a clean foundation for your evaluation set:

- Every selected patient has all of:
  - `patient_overview`
  - `conditions`
  - `medications`
  - `procedures`
  - `observations`
  - `diagnostic_reports`
  - `encounters`
  - `oncology_timeline_events`
  - `oncology_timeline` (plus whatever else is in that set).

So you can safely design a **shared set of question templates** and apply them across all 9 patients, only skipping oncology-specific ones for those who truly have no oncology-flagged content. [youtube](https://www.youtube.com/watch?v=IYx4O42dHd0)

A natural next step, given this:

- Define 3–4 question types that map onto these doc_types (e.g., overview, conditions, medications, oncology history).
- Then, for each of your 9 patients, label gold_chunk_ids for those questions using the doc_type + is_oncology filters you already have.

If you want, I can propose a concrete list of 4 question templates that directly correspond to these doc_types and are suitable for all 9 patients.

Great — here’s a concrete 4-question core set plus a 5-question extended set, both designed to work across all 9 selected patients, along with how to choose `gold_chunk_ids` for each. Because all patients share the same doc_types, you can apply this uniformly. 

### Core 4-question set (recommended baseline)

Use this for your main evaluation; it balances general and oncology content and stays relatively easy to annotate.

1. **Patient overview**

   - Question:  
     “Give a concise overview of this patient’s medical background and current care context.”
   - Primary doc_types to draw gold chunks from:  
     `patient_overview`, `encounters`, `conditions`, `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Include 2–4 chunks that together cover:
       - Key chronic conditions. 
       - Major past procedures or events if they define the patient’s course. 
       - A high-level oncology context if applicable (e.g., diagnosis, line of therapy). 
     - Prefer chunks that are explicitly summary-like (e.g., overview notes) over raw measurements.

2. **Conditions**

   - Question:  
     “What are the patient’s main diagnosed conditions?”
   - Primary doc_types:  
     `conditions`, possibly `encounters` or `patient_overview` if they contain canonical lists. 
   - Gold chunk guidance:
     - Select chunks that list named diagnoses (problem lists, diagnosis sections, structured condition records). 
     - If conditions evolve (e.g., disease progression or resolved conditions), include chunks that clearly mark the current status.

3. **Medications**

   - Question:  
     “What medications is the patient taking or has recently taken?”
   - Primary doc_types:  
     `medications`, optionally `encounters` or `oncology_timeline_events` if they record regimens. 
   - Gold chunk guidance:
     - Focus on chunks that list active or recent meds (lists, med history sections). 
     - If oncology regimens are recorded in timeline events rather than meds, include those chunks too. 

4. **Oncology timeline**

   - Question:  
     “Summarize the patient’s oncology-related timeline, including major events and treatments.”
   - Primary doc_types:  
     `oncology_timeline_events`, `oncology_timeline`, plus supporting `encounters` or `diagnostic_reports` if needed. 
   - Gold chunk guidance:
     - Choose 3–6 chunks covering:
       - Initial diagnosis/first cancer-related event.
       - Key treatment starts/changes (lines of therapy, major procedures). 
       - Notable response/progression events or critical findings (e.g., scan results summaries). 
     - Try to keep the subset coherent and roughly chronological, so a model could reconstruct a timeline from them.

### Extended 5-question set (adds a synthesis question)

For a richer benchmark, add this fifth question; it forces multi-doc-type reasoning.

5. **Cross-document clinical priorities**

   - Question:  
     “What clinical issues appear to be most important for this patient right now?”
   - Primary doc_types:  
     `conditions`, `medications`, `encounters`, `observations`, `diagnostic_reports`, and `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Pick 4–8 chunks such that:
       - Each chunk contributes a distinct piece of evidence (e.g., a condition, a treatment, a key observation/report). 
       - Together they justify why certain issues are “most important” (e.g., active malignancy on treatment, uncontrolled comorbidity, acute complication). 
     - Explicitly favor chunks that are:
       - Recent in time.
       - Clearly interpretable (e.g., summary sections rather than isolated lab values), unless a lab/report is itself critical.

### How to pick gold_chunk_ids in practice

For each patient and each question:

- Step 1: Filter by doc_type and (if you have it) `is_oncology` flag or recency indicators. 
- Step 2: From the filtered subset, manually inspect and choose only the chunks that:
  - Directly answer the question.
  - Are reasonably self-contained (the model doesn’t need many unrelated chunks to interpret them).
- Step 3: Store:
  - `question_text`
  - `patient_id`
  - `gold_chunk_ids` (list of chunk IDs)
  - Optionally a short `rationale` free-text explaining why these chunks were selected (can be handy for later analysis).

This gives you a consistent scheme across all 9 patients: every patient gets the same 4 core questions, and optionally the 5th synthesis question, with gold_chunk_ids drawn from matching doc_types. 

Do you already have a column that marks “recent” vs “historical” chunks (e.g., encounter date), or should I suggest a simple heuristic for recency using only what’s in your `chunks` table?

#### 1. **Patient overview**

This filters candidate chunks for one patient for the Patient Overview question, prioritizing summary-like doc types and creating a readable preview. Filtering with isin(...) and sorting with sort_values(...) are standard Pandas patterns for this kind of review table:

In [ ]:
import pandas as pd
from IPython.display import display

QUESTION_TYPE = "patient_overview"
QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."

overview_doc_types = ["conditions"]
overview_headings = [
    "Recent Condition",
    "Recent Results",
    "Procedures",
]

# Boolean mask for relevant chunks according to your rule
mask_conditions = chunks_df["doc_type"] == "conditions"
mask_headings = chunks_df["heading"].isin(overview_headings)

relevant_mask = mask_conditions | mask_headings

patient_overview_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

patient_overview_gold["question_type"] = QUESTION_TYPE
patient_overview_gold["question_text"] = QUESTION_TEXT

# Optional: reorder columns for clarity
patient_overview_gold = patient_overview_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(patient_overview_gold)